# 03 — Dataset Preparation V2
## Eye Disease Detection — Four-Class Edition

**Target classes:** 0=Normal, 1=Cataract, 2=Diabetic Retinopathy, 3=Glaucoma

Audits all datasets, validates images, detects duplicates, builds master metadata CSV, and creates stratified 70/15/15 train/val/test splits.

**Does NOT train any model. Does NOT modify original datasets.**

In [17]:
# Cell 1 — Imports and configuration
from pathlib import Path
import hashlib
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == 'notebooks' else NOTEBOOK_DIR

DATASET_DIR = PROJECT_DIR / 'dataset'
PREPROC_DIR = PROJECT_DIR / 'preprocessing'
SPLITS_DIR  = PREPROC_DIR / 'splits_v2'
REPORT_DIR  = PROJECT_DIR / 'reports' / 'dataset_v2'

for d in [SPLITS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {0: 'Normal', 1: 'Cataract', 2: 'Diabetic Retinopathy', 3: 'Glaucoma'}

print('Project dir :', PROJECT_DIR)
print('Dataset dir :', DATASET_DIR)
print('Reports dir :', REPORT_DIR)
print('Splits dir  :', SPLITS_DIR)
print('Class map   :', CLASS_MAP)


Project dir : d:\Practice Projects\Disease Detection
Dataset dir : d:\Practice Projects\Disease Detection\dataset
Reports dir : d:\Practice Projects\Disease Detection\reports\dataset_v2
Splits dir  : d:\Practice Projects\Disease Detection\preprocessing\splits_v2
Class map   : {0: 'Normal', 1: 'Cataract', 2: 'Diabetic Retinopathy', 3: 'Glaucoma'}


## Stage 1 — Verify dataset root and list top-level datasets

In [18]:
# Cell 2 — Verify dataset root
assert DATASET_DIR.exists(), f'Dataset dir not found: {DATASET_DIR}'
tops = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
print('Top-level dataset folders:')
for t in tops:
    print(' ', t.name)


Top-level dataset folders:
  Dataset
  Eye Disease detection
  GlaucomaFundusImaging
  Messidor-2+EyePac_Balanced


## Stage 2 — Dataset audit: folder structure, image counts, mapping decisions

In [19]:
# Cell 3 — Audit every dataset folder/class
def count_images(folder):
    return sum(1 for f in folder.rglob('*')
               if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)

audit_rows = []

# Eye Disease detection — direct folder-name labels
edd_map = {
    'cataract': (1, 'Cataract'),
    'diabetic_retinopathy': (2, 'Diabetic Retinopathy'),
    'glaucoma': (3, 'Glaucoma'),
    'normal': (0, 'Normal'),
}
for cls_folder in sorted((DATASET_DIR / 'Eye Disease detection').iterdir()):
    if cls_folder.is_dir():
        cid, cname = edd_map.get(cls_folder.name, (None, None))
        audit_rows.append({
            'source_dataset': 'Eye Disease detection',
            'folder': cls_folder.name,
            'image_count': count_images(cls_folder),
            'reliable_mapping': cid is not None,
            'maps_to_class_id': cid,
            'maps_to_disease': cname,
            'notes': 'Direct folder-name label',
        })

# G1020 — binary label CSV
g1020_df = pd.read_csv(DATASET_DIR / 'GlaucomaFundusImaging' / 'G1020' / 'G1020.csv')
audit_rows.append({'source_dataset': 'GlaucomaFundusImaging/G1020', 'folder': 'Images (label=1)',
    'image_count': int((g1020_df['binaryLabels']==1).sum()), 'reliable_mapping': True,
    'maps_to_class_id': 3, 'maps_to_disease': 'Glaucoma', 'notes': 'G1020.csv binaryLabels=1'})
audit_rows.append({'source_dataset': 'GlaucomaFundusImaging/G1020', 'folder': 'Images (label=0)',
    'image_count': int((g1020_df['binaryLabels']==0).sum()), 'reliable_mapping': True,
    'maps_to_class_id': 0, 'maps_to_disease': 'Normal', 'notes': 'G1020.csv binaryLabels=0'})

# ORIGA — binary label CSV
origa_df = pd.read_csv(DATASET_DIR / 'GlaucomaFundusImaging' / 'ORIGA' / 'OrigaList.csv')
audit_rows.append({'source_dataset': 'GlaucomaFundusImaging/ORIGA', 'folder': 'Images (Glaucoma=1)',
    'image_count': int((origa_df['Glaucoma']==1).sum()), 'reliable_mapping': True,
    'maps_to_class_id': 3, 'maps_to_disease': 'Glaucoma', 'notes': 'OrigaList.csv Glaucoma=1'})
audit_rows.append({'source_dataset': 'GlaucomaFundusImaging/ORIGA', 'folder': 'Images (Glaucoma=0)',
    'image_count': int((origa_df['Glaucoma']==0).sum()), 'reliable_mapping': True,
    'maps_to_class_id': 0, 'maps_to_disease': 'Normal', 'notes': 'OrigaList.csv Glaucoma=0'})

# REFUGE — segmentation only, no per-image class label
refuge_base = DATASET_DIR / 'GlaucomaFundusImaging' / 'REFUGE'
refuge_n = sum(count_images(refuge_base / s / 'Images') for s in ['train','test','val'])
audit_rows.append({'source_dataset': 'GlaucomaFundusImaging/REFUGE', 'folder': 'train+test+val/Images',
    'image_count': refuge_n, 'reliable_mapping': False,
    'maps_to_class_id': None, 'maps_to_disease': None,
    'notes': 'EXCLUDED: segmentation dataset only — no per-image classification label'})

# Dataset — YOLO detection format
for split in ['train', 'test', 'val']:
    audit_rows.append({'source_dataset': 'Dataset', 'folder': split + '/images',
        'image_count': count_images(DATASET_DIR / 'Dataset' / split / 'images'),
        'reliable_mapping': False, 'maps_to_class_id': None, 'maps_to_disease': None,
        'notes': 'EXCLUDED: YOLO detection format (bounding boxes), not a classification dataset'})

# Messidor-2+EyePac_Balanced — DR severity grades 0-4
for cls in ['0','1','2','3','4']:
    audit_rows.append({'source_dataset': 'Messidor-2+EyePac_Balanced', 'folder': cls,
        'image_count': count_images(DATASET_DIR / 'Messidor-2+EyePac_Balanced' / cls),
        'reliable_mapping': False, 'maps_to_class_id': None, 'maps_to_disease': None,
        'notes': f'EXCLUDED: DR severity grade {cls} (0=no DR..4=proliferative) — ambiguous mapping to 4-class schema'})

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(REPORT_DIR / 'dataset_audit_report.csv', index=False)
print(audit_df.to_string(index=False))


              source_dataset                folder  image_count  reliable_mapping  maps_to_class_id      maps_to_disease                                                                                          notes
       Eye Disease detection              cataract         1038              True               1.0             Cataract                                                                       Direct folder-name label
       Eye Disease detection  diabetic_retinopathy         1098              True               2.0 Diabetic Retinopathy                                                                       Direct folder-name label
       Eye Disease detection              glaucoma         1007              True               3.0             Glaucoma                                                                       Direct folder-name label
       Eye Disease detection                normal         1074              True               0.0               Normal                

## Stage 3 — Collect all usable image paths with reliable labels

In [20]:
# Cell 4 — Collect image records from reliable sources only
records = []

# Source 1: Eye Disease detection
edd_map = {
    'cataract': (1, 'Cataract'),
    'diabetic_retinopathy': (2, 'Diabetic Retinopathy'),
    'glaucoma': (3, 'Glaucoma'),
    'normal': (0, 'Normal'),
}
for cls_folder in sorted((DATASET_DIR / 'Eye Disease detection').iterdir()):
    if cls_folder.is_dir() and cls_folder.name in edd_map:
        cid, cname = edd_map[cls_folder.name]
        for img in cls_folder.iterdir():
            if img.is_file() and img.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({'image_path': str(img),
                                'source_dataset': 'Eye Disease detection',
                                'original_label': cls_folder.name,
                                'disease_name': cname, 'class_id': cid})

# Source 2: G1020
g1020_img_dir = DATASET_DIR / 'GlaucomaFundusImaging' / 'G1020' / 'Images'
g1020_df = pd.read_csv(DATASET_DIR / 'GlaucomaFundusImaging' / 'G1020' / 'G1020.csv')
for _, row in g1020_df.iterrows():
    img_path = g1020_img_dir / row['imageID']
    if img_path.exists():
        lbl = int(row['binaryLabels'])
        records.append({'image_path': str(img_path),
                        'source_dataset': 'GlaucomaFundusImaging/G1020',
                        'original_label': str(lbl),
                        'disease_name': 'Glaucoma' if lbl == 1 else 'Normal',
                        'class_id': 3 if lbl == 1 else 0})

# Source 3: ORIGA
origa_img_dir = DATASET_DIR / 'GlaucomaFundusImaging' / 'ORIGA' / 'Images'
origa_df = pd.read_csv(DATASET_DIR / 'GlaucomaFundusImaging' / 'ORIGA' / 'OrigaList.csv')
for _, row in origa_df.iterrows():
    img_path = origa_img_dir / row['Filename']
    if img_path.exists():
        lbl = int(row['Glaucoma'])
        records.append({'image_path': str(img_path),
                        'source_dataset': 'GlaucomaFundusImaging/ORIGA',
                        'original_label': str(lbl),
                        'disease_name': 'Glaucoma' if lbl == 1 else 'Normal',
                        'class_id': 3 if lbl == 1 else 0})

raw_df = pd.DataFrame(records)
print(f'Total raw records collected: {len(raw_df)}')
print(raw_df.groupby(['source_dataset','disease_name','class_id']).size()
      .reset_index(name='count').to_string(index=False))


Total raw records collected: 5887
             source_dataset         disease_name  class_id  count
      Eye Disease detection             Cataract         1   1038
      Eye Disease detection Diabetic Retinopathy         2   1098
      Eye Disease detection             Glaucoma         3   1007
      Eye Disease detection               Normal         0   1074
GlaucomaFundusImaging/G1020             Glaucoma         3    296
GlaucomaFundusImaging/G1020               Normal         0    724
GlaucomaFundusImaging/ORIGA             Glaucoma         3    168
GlaucomaFundusImaging/ORIGA               Normal         0    482


## Stage 4 — Image validation: corrupt and unreadable files

In [21]:
# Cell 5 — Validate images (corrupt / unreadable check)
valid_records = []
corrupted = []

for rec in records:
    try:
        with Image.open(rec['image_path']) as im:
            im.verify()
        valid_records.append(rec)
    except Exception as e:
        corrupted.append({'image_path': rec['image_path'], 'error': str(e)})

print(f'Valid images        : {len(valid_records)}')
print(f'Corrupted/unreadable: {len(corrupted)}')
if corrupted:
    corrupt_df = pd.DataFrame(corrupted)
    corrupt_df.to_csv(REPORT_DIR / 'corrupted_images.csv', index=False)
    print(corrupt_df.to_string(index=False))
else:
    print('No corrupted images found.')


Valid images        : 5887
Corrupted/unreadable: 0
No corrupted images found.


## Stage 5 — Duplicate detection via MD5 hash

In [22]:
# Cell 6 — Detect exact duplicates by MD5 hash
def md5_hash(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

hash_map = {}
dedup_records = []
duplicate_rows = []

for rec in valid_records:
    h = md5_hash(rec['image_path'])
    if h in hash_map:
        duplicate_rows.append({'duplicate_path': rec['image_path'],
                               'original_path': hash_map[h], 'hash': h})
    else:
        hash_map[h] = rec['image_path']
        dedup_records.append({**rec, 'md5': h})

print(f'After dedup — unique images: {len(dedup_records)}')
print(f'Duplicates removed         : {len(duplicate_rows)}')
if duplicate_rows:
    dup_df = pd.DataFrame(duplicate_rows)
    dup_df.to_csv(REPORT_DIR / 'duplicate_images.csv', index=False)
    print(dup_df.head(10).to_string(index=False))
else:
    print('No exact duplicates found.')


After dedup — unique images: 5885
Duplicates removed         : 2
                                                                              duplicate_path                                                                                original_path                             hash
d:\Practice Projects\Disease Detection\dataset\Eye Disease detection\glaucoma\1415_right.jpg d:\Practice Projects\Disease Detection\dataset\Eye Disease detection\cataract\1415_right.jpg e2d70a8954bcdd55ef63989a9a5b84c3
  d:\Practice Projects\Disease Detection\dataset\Eye Disease detection\glaucoma\625_left.jpg   d:\Practice Projects\Disease Detection\dataset\Eye Disease detection\cataract\625_left.jpg 06f4c9b0835adcdddc48252f45f6480e


## Stage 6 — Build master metadata CSV

In [23]:
# Cell 7 — Build master metadata DataFrame
meta_df = pd.DataFrame(dedup_records)[['image_path','source_dataset',
                                         'original_label','disease_name','class_id']]
print('Class distribution before splitting:')
dist = meta_df.groupby(['class_id','disease_name']).size().reset_index(name='count')
print(dist.to_string(index=False))
print(f'\nTotal usable images: {len(meta_df)}')


Class distribution before splitting:
 class_id         disease_name  count
        0               Normal   2280
        1             Cataract   1038
        2 Diabetic Retinopathy   1098
        3             Glaucoma   1469

Total usable images: 5885


## Stage 7 — Stratified 70/15/15 train/validation/test split

In [24]:
# Cell 8 — Stratified split
X = meta_df['image_path'].values
y = meta_df['class_id'].values

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

# 50/50 split of temp => 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

def make_split_df(paths, split_name):
    sub = meta_df.set_index('image_path').loc[paths].reset_index()
    sub['split'] = split_name
    return sub

train_df = make_split_df(X_train, 'train')
val_df   = make_split_df(X_val,   'validation')
test_df  = make_split_df(X_test,  'test')

print(f'Train      : {len(train_df)} images')
print(f'Validation : {len(val_df)} images')
print(f'Test       : {len(test_df)} images')
print(f'Total      : {len(train_df)+len(val_df)+len(test_df)} images')


Train      : 4119 images
Validation : 883 images
Test       : 883 images
Total      : 5885 images


## Stage 8 — Verify splits: no overlap, all classes present

In [25]:
# Cell 9 — Verify no path overlap between splits
train_paths = set(train_df['image_path'])
val_paths   = set(val_df['image_path'])
test_paths  = set(test_df['image_path'])

assert len(train_paths & val_paths)  == 0, 'DATA LEAKAGE: train/val overlap!'
assert len(train_paths & test_paths) == 0, 'DATA LEAKAGE: train/test overlap!'
assert len(val_paths   & test_paths) == 0, 'DATA LEAKAGE: val/test overlap!'
print('No overlap detected — data leakage check PASSED')

for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    classes = set(df['class_id'].unique())
    assert classes == {0,1,2,3}, f'{name} missing classes: {set(range(4))-classes}'
    print(f'{name}: classes present = {sorted(classes)}')
print('All 4 classes present in every split — PASSED')


No overlap detected — data leakage check PASSED
train: classes present = [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
validation: classes present = [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
test: classes present = [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
All 4 classes present in every split — PASSED


## Stage 9 — Save CSV files

In [26]:
# Cell 10 — Save master metadata and split CSVs
COLS = ['image_path','source_dataset','original_label','disease_name','class_id','split']
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)[COLS]

all_df.to_csv(PREPROC_DIR / 'dataset_metadata_v2.csv', index=False)
train_df[COLS].to_csv(SPLITS_DIR / 'train.csv', index=False)
val_df[COLS].to_csv(SPLITS_DIR / 'validation.csv', index=False)
test_df[COLS].to_csv(SPLITS_DIR / 'test.csv', index=False)

print('Saved:')
print(' ', PREPROC_DIR / 'dataset_metadata_v2.csv')
print(' ', SPLITS_DIR / 'train.csv')
print(' ', SPLITS_DIR / 'validation.csv')
print(' ', SPLITS_DIR / 'test.csv')


Saved:
  d:\Practice Projects\Disease Detection\preprocessing\dataset_metadata_v2.csv
  d:\Practice Projects\Disease Detection\preprocessing\splits_v2\train.csv
  d:\Practice Projects\Disease Detection\preprocessing\splits_v2\validation.csv
  d:\Practice Projects\Disease Detection\preprocessing\splits_v2\test.csv


## Stage 10 — Class and split distribution summary

In [27]:
# Cell 11 — Per-class per-split counts table
summary_rows = []
for split_name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    for cid in sorted(CLASS_MAP):
        summary_rows.append({'split': split_name, 'class_id': cid,
                             'disease_name': CLASS_MAP[cid],
                             'count': int((df['class_id'] == cid).sum())})

summary_df = pd.DataFrame(summary_rows)
pivot = summary_df.pivot_table(index=['class_id','disease_name'],
                                columns='split', values='count', aggfunc='sum')
pivot['TOTAL'] = pivot.sum(axis=1)
print(pivot.to_string())
pivot.to_csv(REPORT_DIR / 'split_class_distribution.csv')


split                          test  train  validation  TOTAL
class_id disease_name                                        
0        Normal                 342   1596         342   2280
1        Cataract               156    727         155   1038
2        Diabetic Retinopathy   165    768         165   1098
3        Glaucoma               220   1028         221   1469


In [28]:
# Cell 12 — Bar chart of class distribution per split
fig, ax = plt.subplots(figsize=(10, 5))
splits_order = ['train', 'validation', 'test']
x = list(range(len(CLASS_MAP)))
width = 0.25
colors = ['steelblue', 'darkorange', 'green']
for i, (sname, color) in enumerate(zip(splits_order, colors)):
    counts = [int(summary_df[(summary_df.split==sname) &
                              (summary_df.class_id==cid)]['count'].values[0])
              for cid in sorted(CLASS_MAP)]
    ax.bar([xi + i*width for xi in x], counts, width, label=sname, color=color, alpha=0.85)
ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels([CLASS_MAP[c] for c in sorted(CLASS_MAP)], fontsize=11)
ax.set_ylabel('Image count')
ax.set_title('Class distribution per split — Dataset V2')
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / 'class_distribution_v2.png', dpi=120)
plt.show()
print('Chart saved to', REPORT_DIR / 'class_distribution_v2.png')


Chart saved to d:\Practice Projects\Disease Detection\reports\dataset_v2\class_distribution_v2.png


C:\Users\NTS0287\AppData\Local\Temp\ipykernel_42208\3963268574.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Stage 11 — Per-source breakdown report

In [29]:
# Cell 13 — Per-source per-class split detail
detail_rows = []
for src, grp in all_df.groupby('source_dataset'):
    for cid, cgrp in grp.groupby('class_id'):
        detail_rows.append({
            'source_dataset': src,
            'disease_name': CLASS_MAP[cid],
            'class_id': cid,
            'total': len(cgrp),
            'train': int((cgrp['split']=='train').sum()),
            'validation': int((cgrp['split']=='validation').sum()),
            'test': int((cgrp['split']=='test').sum()),
        })
detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv(REPORT_DIR / 'source_class_split_detail.csv', index=False)
print(detail_df.to_string(index=False))


             source_dataset         disease_name  class_id  total  train  validation  test
      Eye Disease detection               Normal         0   1074    762         151   161
      Eye Disease detection             Cataract         1   1038    727         155   156
      Eye Disease detection Diabetic Retinopathy         2   1098    768         165   165
      Eye Disease detection             Glaucoma         3   1005    681         161   163
GlaucomaFundusImaging/G1020               Normal         0    724    502         114   108
GlaucomaFundusImaging/G1020             Glaucoma         3    296    225          33    38
GlaucomaFundusImaging/ORIGA               Normal         0    482    332          77    73
GlaucomaFundusImaging/ORIGA             Glaucoma         3    168    122          27    19


## DATASET PREPARATION V2 — COMPLETED

In [30]:
# Cell 14 — Final summary
total_usable   = len(all_df)
total_excluded = (len(records) - len(valid_records)) + len(duplicate_rows)

print('=' * 62)
print('  DATASET PREPARATION V2 COMPLETED')
print('=' * 62)
print(f'  Total usable images  : {total_usable}')
print(f'  Corrupted excluded   : {len(corrupted)}')
print(f'  Duplicates excluded  : {len(duplicate_rows)}')
print(f'  Total excluded       : {total_excluded}')
print()
print('  Class-wise counts (all splits combined):')
for cid in sorted(CLASS_MAP):
    n = int((all_df['class_id'] == cid).sum())
    print(f'    {cid} — {CLASS_MAP[cid]:<22}: {n}')
print()
print('  Split-wise counts:')
for sname in ['train', 'validation', 'test']:
    n = int((all_df['split'] == sname).sum())
    print(f'    {sname:<12}: {n}')
print()
print('  Intentionally excluded datasets/classes:')
print('    - GlaucomaFundusImaging/REFUGE   : segmentation only, no classification label')
print('    - Dataset (YOLO)                 : detection format, not classification')
print('    - Messidor-2+EyePac_Balanced 0-4 : DR severity grades cannot be reliably')
print('      mapped to our 4-class schema without conflating Normal vs DR severity')
print()
print('  Output files:')
print(f'    preprocessing/dataset_metadata_v2.csv')
print(f'    preprocessing/splits_v2/train.csv')
print(f'    preprocessing/splits_v2/validation.csv')
print(f'    preprocessing/splits_v2/test.csv')
print(f'    reports/dataset_v2/ (audit reports + chart)')
print('=' * 62)


  DATASET PREPARATION V2 COMPLETED
  Total usable images  : 5885
  Corrupted excluded   : 0
  Duplicates excluded  : 2
  Total excluded       : 2

  Class-wise counts (all splits combined):
    0 — Normal                : 2280
    1 — Cataract              : 1038
    2 — Diabetic Retinopathy  : 1098
    3 — Glaucoma              : 1469

  Split-wise counts:
    train       : 4119
    validation  : 883
    test        : 883

  Intentionally excluded datasets/classes:
    - GlaucomaFundusImaging/REFUGE   : segmentation only, no classification label
    - Dataset (YOLO)                 : detection format, not classification
    - Messidor-2+EyePac_Balanced 0-4 : DR severity grades cannot be reliably
      mapped to our 4-class schema without conflating Normal vs DR severity

  Output files:
    preprocessing/dataset_metadata_v2.csv
    preprocessing/splits_v2/train.csv
    preprocessing/splits_v2/validation.csv
    preprocessing/splits_v2/test.csv
    reports/dataset_v2/ (audit reports + 